# Tier 2 on Kaggle: 2 x T4

**This notebook has not yet been executed on a real Kaggle session.**
When someone runs it end to end successfully, delete this paragraph and say so
in the commit message. An untested notebook is usually a broken notebook, and
claiming otherwise wastes someone an afternoon.

---

## Before you start

1. Complete any account verification Kaggle currently requires for accelerators.
2. If the right-hand Accelerator panel offers `GPU T4 x 2`, select it. The
   notebook verifies the actual device count before launching Tier 2.
3. **Right-hand panel, Internet, On.** Needed to clone the repo.

## Why this notebook exists

Tiers 0 and 1 run on your laptop and answer *what happens*. They cannot answer
*how fast*: several CPU processes on one machine share a memory bus and a core
count, so adding ranks adds communication without adding compute.

Two CUDA devices with a real interconnect can answer it. Kaggle has historically
offered a two-T4 option, but availability and quotas vary.

## Two things about T4s

**No bf16 tensor cores.** T4s are Turing generation. Use fp16 or fp32, and gate
on `torch.cuda.is_bf16_supported()` rather than assuming.

**Verify the topology.** Do not assume NVLink or a fixed peer-to-peer bandwidth
from the accelerator label. Run `nvidia-smi topo -m` and treat the assigned
host's topology and measured bandwidth as part of the result.


## 1. Setup


In [ ]:
!nvidia-smi

# Must show two devices. One means the accelerator setting did not take:
# change it in the right-hand panel and restart the session.


In [ ]:
import pathlib
import subprocess

REPO_URL = ''  # Paste this repository's public HTTPS clone URL.
if not REPO_URL.startswith('https://'):
    raise ValueError('Set REPO_URL to the public HTTPS Git clone URL first')

checkout = pathlib.Path('/kaggle/working/dtfs')
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(checkout)], check=True)
%cd /kaggle/working/dtfs


In [ ]:
# Install the repo WITHOUT touching torch.
#
# First verify the runtime's existing CUDA-enabled torch build. Replacing a
# compatible build consumes session time and can install a wheel mismatched
# to the runtime. --no-deps installs this package and nothing else; the next
# command installs every non-Torch runtime and test dependency explicitly.
!pip install -q -e . --no-deps
!python -m pip install -q 'numpy>=1.26' 'pyyaml>=6.0' 'matplotlib>=3.8' 'psutil>=5.9' 'setuptools>=83.0' 'pytest>=8.0' 'pytest-timeout>=2.3'
!python -m pip check


In [ ]:
!python scripts/doctor.py

# The verdict must say:  Tier 2 (NCCL multi-GPU):  AVAILABLE
# If it does not, stop here. Nothing below will work, and the failure mode
# is a hang rather than an error.


## 2. Pick a lab

Change `LAB` and re-run from here. Everything below reads it.


In [ ]:
LAB = 0   # benchmarks: 0, 1, '1b', 2, 3, 4; Tier-2 tests: 0, 1, 2, 3

BENCH = {
    0:    'labs/lab0_collectives/bench/sweep.py',
    1:    'labs/lab1_ddp_bucketing/bench/bucket_sweep.py',
    '1b': 'labs/lab1b_zero_memory/bench/memory_curve.py',
    2:    'labs/lab2_tensor_parallel/bench/tp_scaling.py',
    3:    'labs/lab3_pipeline_parallel/bench/bubble_sweep.py',
    4:    'labs/lab4_capstone_2d/bench/mfu.py',
}

LAB_DIR = {
    0:    'labs/lab0_collectives',
    1:    'labs/lab1_ddp_bucketing',
    '1b': 'labs/lab1b_zero_memory',
    2:    'labs/lab2_tensor_parallel',
    3:    'labs/lab3_pipeline_parallel',
    4:    'labs/lab4_capstone_2d',
}
TIER2_LABS = {0, 1, 2, 3}

print(f'lab {LAB}: {BENCH[LAB]}')


## 3. Bring your own work

The repository ships incomplete exercise files under `labs/*/starter/` and
completed published implementations under `answers/`.

Three ways to choose what this temporary session runs:

**A. Run the published reference.** Copy the matching `answers/<lab>/starter/`
files over `labs/<lab>/starter/` in this disposable clone.

**B. Push your exercise work to a fork** and clone that instead of the upstream
URL. Best if you are working over several sessions.

**C. Upload your `workspace/` as a Kaggle Dataset or use `%%writefile`.** Fine
for a one-off; copy the files into the matching `labs/<lab>/starter/` path.

The included tests and benchmarks already spawn their worker processes through
`common.dist_utils.run_distributed`. Run them with plain `pytest` or `python`.
Wrapping them in `torchrun` creates a second, conflicting launch layer. For your
own spawn-based entry point, write it to a `.py` file: cell-defined functions do
not pickle cleanly, and `fork` plus CUDA is unsafe.


In [ ]:
import pathlib
import shutil

# Option A is the default. Set this to False before testing work from your fork
# or an attached dataset.
USE_PUBLISHED_ANSWERS = True
if USE_PUBLISHED_ANSWERS:
    lab_name = pathlib.Path(LAB_DIR[LAB]).name
    source = pathlib.Path('answers') / lab_name / 'starter'
    target = pathlib.Path(LAB_DIR[LAB]) / 'starter'
    shutil.copytree(source, target, dirs_exist_ok=True)
    print('overlaid published answer:', source)

# Option C: copy your work in from an attached dataset.
# import shutil, pathlib
# src = pathlib.Path('/kaggle/input/my-dtfs-answers')
# for f in src.rglob('*.py'):
#     dst = pathlib.Path('/kaggle/working/dtfs') / f.relative_to(src)
#     dst.parent.mkdir(parents=True, exist_ok=True)
#     shutil.copy(f, dst)
#     print('copied', dst)


## 4. Correctness first

Tiers 0 and 1 run here too, on CPU, and they are far faster to debug than a
NCCL job. Get them green before spending GPU-hours on a timing run.


In [ ]:
!python -m pytest {LAB_DIR[LAB]} -m 'not tier2' -q --timeout=300


## 5. Tier 2

Labs 0, 1, 2 and 3 contain tests marked `tier2`. Lab 1B and Lab 4 have GPU
benchmarks but no Tier-2 pytest cases, so this cell reports that distinction
instead of invoking pytest with an empty selection.


In [ ]:
import subprocess
import sys

if LAB in TIER2_LABS:
    subprocess.run(
        [sys.executable, '-m', 'pytest', LAB_DIR[LAB], '-m', 'tier2', '-v', '--timeout=900'],
        check=True,
    )
else:
    print(f'Lab {LAB} has no Tier-2 pytest cases; continue to its GPU benchmark.')


## 6. The benchmark

This driver calls `run_distributed` and owns its worker processes. Run it once
with plain Python; do not wrap it in `torchrun`.


In [ ]:
!python {BENCH[LAB]} --nproc 2


## 7. NCCL algorithm selection

Lab 0 asks you to compare simplified ring and tree cost models. NCCL's runtime
selector can consider those and other algorithms and protocols, and its tuning
log tells you what it chose for each message size on this assigned topology.

Run this, then force each algorithm with `NCCL_ALGO` and compare the timings
against your fitted two-algorithm crossover. Treat the result as one topology's
measurement rather than a universal selection rule.


In [ ]:
%%writefile /kaggle/working/nccl_probe.py
import os
import time

import torch
import torch.distributed as dist

local_rank = int(os.environ['LOCAL_RANK'])
torch.cuda.set_device(local_rank)
dist.init_process_group('nccl')
rank = dist.get_rank()
world = dist.get_world_size()

for exponent in range(12, 27, 2):
    n_bytes = 1 << exponent
    tensor = torch.ones(n_bytes // 4, device=f'cuda:{rank}')

    for _ in range(5):
        dist.all_reduce(tensor)
    torch.cuda.synchronize()

    start = time.perf_counter()
    for _ in range(20):
        dist.all_reduce(tensor)
    torch.cuda.synchronize()
    seconds = (time.perf_counter() - start) / 20

    if rank == 0:
        algo_bw = n_bytes / seconds / 1e9
        bus_bw = 2 * (world - 1) / world * n_bytes / seconds / 1e9
        print(f'{n_bytes / 1e6:8.2f} MB  {seconds * 1e3:8.3f} ms  '
              f'algo {algo_bw:6.2f} GB/s  bus {bus_bw:6.2f} GB/s')

dist.destroy_process_group()


In [ ]:
# This probe is rank-aware top-level code, so torchrun is appropriate here.
!NCCL_DEBUG=INFO NCCL_DEBUG_SUBSYS=TUNING torchrun --standalone --nnodes=1 --nproc_per_node=2 /kaggle/working/nccl_probe.py 2>&1 | grep -Ei 'MB|Ring|Tree|algo|proto' | head -40


In [ ]:
print('=== forced Ring ===')
!NCCL_ALGO=Ring torchrun --standalone --nnodes=1 --nproc_per_node=2 /kaggle/working/nccl_probe.py
print()
print('=== forced Tree ===')
!NCCL_ALGO=Tree torchrun --standalone --nnodes=1 --nproc_per_node=2 /kaggle/working/nccl_probe.py


## 8. Reconcile


In [ ]:
!python -m common.predictions --lab {LAB_DIR[LAB]}


## 9. Get your artefacts out

The session ends and takes `/kaggle/working` with it unless you save. Commit
the notebook, which snapshots the output directory, or download the zip.


In [ ]:
import shutil

shutil.make_archive('/kaggle/working/runs', 'zip', '/kaggle/working/dtfs/runs')
print('download /kaggle/working/runs.zip from the Output panel')

!ls -la /kaggle/working/dtfs/runs/


---

## Housekeeping

**Quota and hardware vary.** Read the account's current quota display and verify
the assigned devices with `nvidia-smi`. Debug Tier 0 and Tier 1 without an
accelerator when possible.

**Hosted sessions are temporary.** Checkpoint anything long or expensive.

**Nothing here touches your machine.** The whole repo lives in
`/kaggle/working` and is normally discarded with the session. Download any
artefacts you need before the session ends.
